# Notebook 09 — Final Hybrid Submission

**Purpose:** Build the final submission file using the best validated model for each brand.

**Model assignment (evidence-based — validated on Jul-Dec 2024):**

| Brand | Model | Validation WAPE | Reason |
|-------|-------|----------------|--------|
| Xolarin | TiDE v5 | 0.66% | Stable, new features added noise |
| Hemvia | TiDE v5 | 0.80% | Stable, best as-is |
| Ocretiva | TiDE v5 | 0.93% | Payer break already captured |
| Retivue | TiDE v6 | 5.13% | Structural features marginally helped |
| Vabyseal | TiDE v5 | 6.41% | v8 was worse for this brand |
| Perjenta | TiDE v5 | 5.50% | v8 was worse for this brand |
| Kadcynex | TiDE v8 + ETS 50/50 | 6.86% | prior_auth_delta + ETS brand anchor |
| Phesgrox | TiDE v8 + Prophet 75/25 | 7.07% | Ensemble corrects seasonal under-prediction |

**Overall WAPE: ~3.12% (H2 2024) | ~3.59% (H1 2024 backtest)**
**vs TM1 baseline: 77% better**

**Run time:** <5 minutes

## Step 1 — Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / '03_scripts'))

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from utils import make_submission

OUTPUT = Path('../04_outputs/final')
OUTPUT.mkdir(exist_ok=True)

# Check all required model submissions exist
required = {
    'TiDE v5': '../04_outputs/tide/tide_v5_submission.csv',
    'TiDE v6': '../04_outputs/tide/tide_v6_submission.csv',
    'Ensemble': '../04_outputs/ensemble/ensemble_final_submission.csv',
}
print('Checking required files...')
for name, path in required.items():
    status = '✅' if Path(path).exists() else '❌ MISSING'
    print(f'  {name:<12}: {status}')

## Step 2 — Build Smart Hybrid

In [ ]:
import subprocess
project = Path('..').resolve()
result  = subprocess.run(
    [sys.executable, str(project / '03_scripts' / 'run_smart_hybrid.py')],
    cwd=str(project), capture_output=False
)
print('Done!' if result.returncode == 0 else f'Error: {result.returncode}')

## Step 3 — Verify Final Submission

In [ ]:
sub = pd.read_csv('../04_outputs/final/final_submission.csv')
test_meta = pd.read_csv('../01_input/raw/test_features.csv')
sub = sub.merge(test_meta[['row_id','product_brand_name']], on='row_id', how='left')

print(f'Total rows     : {len(sub):,}  (expected 3,840)')
print(f'Null forecasts : {sub["forecast_units_eqv"].isna().sum()}  (must be 0)')
print(f'Negative values: {(sub["forecast_units_eqv"]<0).sum()}  (must be 0)')
print(f'Min forecast   : {sub["forecast_units_eqv"].min():.2f}')
print(f'Max forecast   : {sub["forecast_units_eqv"].max():.2f}')
print()
print('Forecasts by brand:')
print(sub.groupby('product_brand_name')['forecast_units_eqv'].agg(['mean','min','max']).round(1).to_string())
print()
print('✅ Ready to submit: 04_outputs/final/final_submission.csv')

## Step 4 — Expected WAPE Summary

In [ ]:
final_wapes = {
    'Xolarin':  0.0066, 'Hemvia':   0.0080, 'Ocretiva': 0.0093,
    'Retivue':  0.0513, 'Vabyseal': 0.0641, 'Perjenta': 0.0550,
    'Kadcynex': 0.0686, 'Phesgrox': 0.0707,
}
import numpy as np

print(f'{"Brand":<12} {"WAPE":>8} {"vs TM1":>10} {"Status"}')
print('-'*50)
for brand, w in sorted(final_wapes.items(), key=lambda x: x[1]):
    vs_tm1 = (0.137-w)/0.137*100
    status = '✅ <5%' if w<0.05 else ('⚠️ 5-7%' if w<0.07 else '⚠️ >7%')
    print(f'  {brand:<12} {w*100:>6.2f}%  {vs_tm1:>+8.1f}%  {status}')

macro = np.mean(list(final_wapes.values()))
print(f'\n  MACRO-WAPE : {macro*100:.2f}%  (TM1: 13.70%)')
print(f'  Improvement: {(0.137-macro)/0.137*100:.1f}% better than TM1')
print()
print('H1 2024 Backtest: 3.59%  |  H2 2024 Validation: 3.12%')
print('Difference: 0.47% — model is consistent across the full year')